# 4-Term ILP — Toy Problem Analysis

Runs the C2 dispatch pipeline on a single CCP → multiple R1 scenario,
comparing `make_ilp_4term` under different weight configurations against the original ILP and a myopic baseline.

```
score(r, a, f) = M
               + α · triage_value(r)          T1=3, T2=2, T3=1
               + β · sojourn(r)               now − t_issued
               − γ · transport_time(r, a, f)  d(amb→ccp) + d(ccp→facility)
               − δ · operation_time(r, f)     expected wait if joined now
```

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Callable
from pulp import LpProblem, LpVariable, LpMaximize, lpSum, value, PULP_CBC_CMD

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'font.family': 'sans-serif'})

## Stubs & Scenario

In [ ]:
@dataclass
class Patient:
    patient_id: str
    triage: str

@dataclass
class Unit:
    _wait: float = 0.0
    def expected_wait_if_join_now(self) -> float:
        return self._wait

@dataclass
class Facility:
    loc_id: str
    units: Dict[str, Unit] = field(default_factory=dict)
    exit_queue: list = field(default_factory=list)

@dataclass
class Ambulance:
    amb_id: str
    capacity: int
    location: 'Facility'
    plan: list = field(default_factory=list)
    def _loc_id(self, loc) -> str:
        return loc.loc_id if loc is not None else ''

@dataclass
class PlanStep:
    facility: Facility
    unload_patients: list
    load_patients: list

@dataclass
class TransportRequest:
    req_id: str
    patient: Patient
    issuing_facility: Facility
    t_issued: float
    injury_time: float
    triage: str
    needed_unit: Optional[str]
    status: str = 'pending'

    @property
    def casualty_id(self): return self.patient.patient_id
    @property
    def issuing_facility_id(self): return self.issuing_facility.loc_id

class Env:
    def __init__(self, now: float = 100.0):
        self._now = now
        self.fac_eligibility_datasets: Dict[int, pd.DataFrame] = {}
        self.amb_eligibility_datasets: Dict[int, pd.DataFrame] = {}
        self.facility_by_id: Dict[str, Facility] = {}
        self.amb_by_id: Dict[str, Ambulance] = {}
        self.distance_matrix: Optional[pd.DataFrame] = None
    def now(self) -> float: return self._now

In [ ]:
# ── Facilities ────────────────────────────────────────────────────────────────
# r1_alpha: closest but heavily clogged
# r1_bravo: medium distance, moderate wait
# r1_charlie: far but free
# r1_delta:  medium distance, light wait
ccp        = Facility(loc_id='ccp',        units={})
r1_alpha   = Facility(loc_id='r1_alpha',   units={'treatment': Unit(45.0)})  # closest, most clogged
r1_bravo   = Facility(loc_id='r1_bravo',   units={'treatment': Unit(15.0)})
r1_charlie = Facility(loc_id='r1_charlie', units={'treatment': Unit( 0.0)})  # farthest, completely free
r1_delta   = Facility(loc_id='r1_delta',   units={'treatment': Unit( 5.0)})

ALL_FACILITIES = [ccp, r1_alpha, r1_bravo, r1_charlie, r1_delta]

# ── 10 patients: T3 arrive first (longest sojourn), T1 arrive last ────────────
#   t_issued:  T3 issued at t=40-55, T2 at t=65-75, T1 at t=85-95
#   at env.now()=100, T3 sojourn ≈ 45-60 min, T1 sojourn ≈ 5-15 min
_triages  = ['T3','T3','T3','T2','T2','T2','T2','T1','T1','T1']
_t_issued = [ 40,  45,  55,  65,  68,  72,  75,  85,  90,  95]
patients  = [Patient(f'p{i:03d}', t) for i, t in enumerate(_triages, 1)]

ALL_REQUESTS = [
    TransportRequest(
        req_id=f'req_{i:03d}', patient=patients[i-1],
        issuing_facility=ccp,
        t_issued=float(_t_issued[i-1]), injury_time=float(_t_issued[i-1]-5),
        triage=patients[i-1].triage, needed_unit='treatment',
    )
    for i in range(1, 11)
]

# ── Ambulances ────────────────────────────────────────────────────────────────
amb_1 = Ambulance(amb_id='amb_1', capacity=2, location=ccp)
amb_2 = Ambulance(amb_id='amb_2', capacity=2, location=r1_alpha)
ALL_AMBULANCES = [amb_1, amb_2]

# ── Distance matrix (travel time in minutes) ─────────────────────────────────
fac_ids = ['ccp', 'r1_alpha', 'r1_bravo', 'r1_charlie', 'r1_delta']
distance_matrix = pd.DataFrame([
    [  0,  10,  25,  40,  20],   # r1_alpha is closest (10 min)
    [ 10,   0,  20,  35,  15],
    [ 25,  20,   0,  30,  15],
    [ 40,  35,  30,   0,  35],
    [ 20,  15,  15,  35,   0],
], index=fac_ids, columns=fac_ids)

# ── Eligibility matrices ──────────────────────────────────────────────────────
fac_elig_0 = pd.DataFrame([
    [0, 1, 1, 1, 1],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
], index=fac_ids, columns=fac_ids)

amb_elig_0 = pd.DataFrame([
    [1, 1, 1, 1, 1],
    [1, 1, 1, 1, 1],
], index=['amb_1', 'amb_2'], columns=fac_ids)

# ── Build env ─────────────────────────────────────────────────────────────────
env = Env(now=100.0)
env.fac_eligibility_datasets = {0: fac_elig_0}
env.amb_eligibility_datasets = {0: amb_elig_0}
env.distance_matrix          = distance_matrix
env.facility_by_id           = {f.loc_id: f for f in ALL_FACILITIES}
env.amb_by_id                = {a.amb_id: a for a in ALL_AMBULANCES}

def reset_scenario():
    for req in ALL_REQUESTS: req.status = 'pending'
    ccp.exit_queue = list(ALL_REQUESTS)
    for amb in ALL_AMBULANCES: amb.plan = []

reset_scenario()

# ── 2-D positions for visualization ──────────────────────────────────────────
_SCALE   = 0.25
_ANGLES  = {'r1_alpha': 15, 'r1_bravo': -40, 'r1_charlie': 70, 'r1_delta': -75}
POSITIONS = {'ccp': (0.0, 0.0)}
for _fid, _deg in _ANGLES.items():
    _rad = np.radians(_deg)
    _d   = distance_matrix.loc['ccp', _fid]
    POSITIONS[_fid] = (_d * _SCALE * np.cos(_rad), _d * _SCALE * np.sin(_rad))

print('Facilities:', [(f.loc_id, list(f.units.values())[0]._wait if f.units else '-') for f in ALL_FACILITIES])
print('Ambulances:', [a.amb_id for a in ALL_AMBULANCES])
print('Requests  :')
for r in ALL_REQUESTS:
    print(f'  {r.req_id}  {r.triage}  t_issued={r.t_issued:.0f}  sojourn={env.now()-r.t_issued:.0f} min')

## Pipeline Functions

In [ ]:
def snapshot_facilities(facilities):
    if not facilities: return pd.DataFrame()
    all_st = sorted({st for fac in facilities for st in fac.units})
    rows = []
    for fac in facilities:
        row = {'facility_id': fac.loc_id}
        for st in all_st:
            u = fac.units.get(st)
            row[st] = float(u.expected_wait_if_join_now()) if u else False
        rows.append(row)
    return pd.DataFrame(rows)


def build_pending_df(requests):
    return pd.DataFrame([{
        'req_id': r.req_id, 'casualty_id': r.casualty_id,
        'triage': r.triage, 'issuing_facility_id': r.issuing_facility.loc_id,
        'injury_time': r.injury_time, 't_issued': r.t_issued,
        'needed_unit': r.needed_unit, 'status': r.status,
    } for r in requests])


def fac_eligibility(env, requests_df, lever_value):
    df = requests_df.copy()
    df['eligible_facilities'] = [[] for _ in range(len(df))]
    if df.empty: return df
    matrix = (env.fac_eligibility_datasets or {}).get(lever_value)
    if matrix is None or matrix.empty: return df
    def get_eligible(fid):
        if fid not in matrix.index: return []
        return [c for c in matrix.columns if matrix.loc[fid, c] == 1]
    df['eligible_facilities'] = df['issuing_facility_id'].apply(get_eligible)
    return df


def amb_eligibility(env, requests_df, idle_ambulances, lever_value):
    df       = requests_df.copy()
    matrix   = (env.amb_eligibility_datasets or {}).get(lever_value)
    idle_ids = [a.amb_id for a in idle_ambulances]

    def get_eligible(origin):
        if matrix is None or matrix.empty:
            return idle_ids
        return [
            amb_id for amb_id in idle_ids
            if amb_id in matrix.index
            and origin in matrix.columns
            and matrix.loc[amb_id, origin] == 1
        ]

    df['eligible_ambulances'] = df['issuing_facility_id'].apply(get_eligible)
    return df


def craft_plans(env, assigned_df):
    plans = {}
    if assigned_df is None or assigned_df.empty: return plans
    df = assigned_df.dropna(subset=['assigned_amb_id', 'assigned_facility_id']).copy()
    for amb_id, g in df.groupby('assigned_amb_id', sort=False):
        amb = env.amb_by_id.get(str(amb_id))
        if amb is None: continue
        pickup_fac_id  = g['issuing_facility_id'].value_counts().index[0]
        gg = g[g['issuing_facility_id'] == pickup_fac_id].sample(frac=1, random_state=42).reset_index(drop=True)
        gg = gg.iloc[:min(int(amb.capacity), len(gg))]
        dropoff_fac_id = gg['assigned_facility_id'].value_counts().index[0]
        pickup_fac  = env.facility_by_id.get(pickup_fac_id)
        dropoff_fac = env.facility_by_id.get(dropoff_fac_id)
        if pickup_fac is None or dropoff_fac is None: continue
        rmap     = {getattr(r, 'req_id', None): r for r in getattr(pickup_fac, 'exit_queue', [])}
        req_objs = [rmap.get(rid) for rid in gg['req_id'].astype(str).tolist()]
        req_objs = [r for r in req_objs if r is not None and r.patient is not None]
        patients = [r.patient for r in req_objs]
        if not patients: continue
        for r in req_objs: r.status = 'assigned'
        plans[str(amb_id)] = [
            PlanStep(facility=pickup_fac,  unload_patients=[],      load_patients=patients),
            PlanStep(facility=dropoff_fac, unload_patients=patients, load_patients=[]),
        ]
    return plans


def dispatch_ambulances(env, idle_ambulances, pending_df, plans_craft):
    pending_df = pending_df.copy()
    for amb_id, plan in (plans_craft or {}).items():
        amb = env.amb_by_id.get(amb_id)
        if amb is None or amb not in idle_ambulances: continue
        amb.plan = plan
        idle_ambulances.remove(amb)
        loaded = {p.patient_id for step in plan for p in step.load_patients}
        if loaded:
            mask = (pending_df['status'] == 'pending') & \
                   (pending_df['casualty_id'].astype(str).isin({str(x) for x in loaded}))
            pending_df.loc[mask, 'status'] = 'assigned'
    return pending_df


def run_c2_cycle(
    env, requests,
    lever_fac_elig=0, lever_amb_elig=0,
    alpha=1.0, beta=0.0, gamma=0.0, delta=0.0,
    random_seed=42,
):
    idle_ambulances = [a for a in env.amb_by_id.values() if not a.plan]
    pending_df      = build_pending_df(requests)
    requests_df     = pending_df[pending_df['status'] == 'pending'].copy()
    elig_df         = fac_eligibility(env, requests_df, lever_value=lever_fac_elig)
    elig_df         = amb_eligibility(env, elig_df, idle_ambulances, lever_value=lever_amb_elig)
    assigned_df     = solve_ilp(env, elig_df, idle_ambulances, env.distance_matrix,
                                alpha, beta, gamma, delta, random_seed=random_seed)
    plans           = craft_plans(env, assigned_df)
    final_df        = dispatch_ambulances(env, idle_ambulances, pending_df, plans)
    return {'pending_df': final_df, 'assigned_df': assigned_df, 'plans': plans}

print('Pipeline functions loaded.')

## Assignment Policies

In [ ]:
def solve_ilp(
    env, requests_df, idle_ambulances, distance_matrix,
    alpha=1.0, beta=0.0, gamma=0.0, delta=0.0,
    M=10_000.0, random_seed=42,
):
    """
    Assigns patients to ambulances and facilities.

    Expects requests_df to have:
      - eligible_facilities:  list of facility IDs (from fac_eligibility)
      - eligible_ambulances:  list of ambulance IDs (from amb_eligibility)

    score(r,a,f) = M
                 + alpha * triage_value(r)     T1=3, T2=2, T3=1
                 + beta  * sojourn(r)          now - t_issued
                 - gamma * transport_time      d(amb→origin) + d(origin→facility)
                 - delta * operation_time      EWT at destination

    Constraint: all patients on the same ambulance share one destination (y[a,f]).
    """
    _TV = {'T1': 3.0, 'T2': 2.0, 'T3': 1.0}

    df = requests_df.copy()
    if df.empty:
        df['assigned_facility_id'] = None
        df['assigned_amb_id'] = None
        return df

    fac_snap  = snapshot_facilities(list(env.facility_by_id.values()))
    fac_index = fac_snap.set_index('facility_id') if not fac_snap.empty else pd.DataFrame()
    req_idx   = df.index.tolist()
    caps      = {a.amb_id: int(getattr(a, 'capacity', 1)) for a in idle_ambulances}

    prob  = LpProblem('ilp', LpMaximize)
    x, costs = {}, {}

    for idx, row in df.iterrows():
        eligible_facs = row.get('eligible_facilities') or []
        eligible_ambs = row.get('eligible_ambulances') or []
        origin        = row['issuing_facility_id']
        unit_type     = row.get('needed_unit')
        triage_val    = _TV.get(str(row.get('triage', '')), 1.0)
        sojourn       = max(env.now() - float(row.get('t_issued', env.now())), 0.0)

        for amb_id in eligible_ambs:
            amb = env.amb_by_id.get(str(amb_id))
            if amb is None: continue

            amb_loc  = amb._loc_id(amb.location)
            d_pickup = (float(distance_matrix.loc[amb_loc, origin])
                        if distance_matrix is not None
                        and amb_loc in distance_matrix.index
                        and origin  in distance_matrix.columns else 0.0)

            for fac_id in eligible_facs:
                d_transport = (float(distance_matrix.loc[origin, fac_id])
                               if distance_matrix is not None
                               and origin in distance_matrix.index
                               and fac_id in distance_matrix.columns else 0.0)

                ewt = 0.0
                if (not fac_index.empty and fac_id in fac_index.index
                        and unit_type and unit_type in fac_index.columns):
                    val = fac_index.at[fac_id, unit_type]
                    if val is not False and not (isinstance(val, float) and pd.isna(val)):
                        ewt = float(val)

                key = (idx, amb_id, fac_id)
                x[key]     = LpVariable(f'x_{idx}_{amb_id}_{fac_id}', cat='Binary')
                costs[key] = (M + alpha * triage_val + beta * sojourn
                              - gamma * (d_pickup + d_transport) - delta * ewt)

    if not x:
        df['assigned_facility_id'] = None
        df['assigned_amb_id'] = None
        return df

    # y[a, f] = 1 if ambulance a is dispatched to facility f
    af_pairs = set((amb_id, fac_id) for (_, amb_id, fac_id) in x)
    y = {(amb_id, fac_id): LpVariable(f'y_{amb_id}_{fac_id}', cat='Binary')
         for (amb_id, fac_id) in af_pairs}

    prob += lpSum(costs[k] * x[k] for k in x)

    # Each request assigned at most once
    for idx in req_idx:
        vars_r = [x[k] for k in x if k[0] == idx]
        if vars_r: prob += lpSum(vars_r) <= 1

    # Each ambulance respects capacity
    for amb_id in caps:
        vars_a = [x[k] for k in x if k[1] == amb_id]
        if vars_a: prob += lpSum(vars_a) <= caps[amb_id]

    # Single destination per ambulance
    for (amb_id, fac_id), yvar in y.items():
        vars_xaf = [x[k] for k in x if k[1] == amb_id and k[2] == fac_id]
        if vars_xaf:
            prob += lpSum(vars_xaf) <= caps[amb_id] * yvar

    for amb_id in caps:
        vars_y = [y[k] for k in y if k[0] == amb_id]
        if vars_y: prob += lpSum(vars_y) <= 1

    prob.solve(PULP_CBC_CMD(msg=0, options=[f'RandomS {random_seed}']))

    af = {i: None for i in req_idx}
    aa = {i: None for i in req_idx}
    for (idx, amb_id, fac_id), var in x.items():
        if (value(var) or 0) > 0.5:
            af[idx] = fac_id
            aa[idx] = amb_id

    df['assigned_facility_id'] = [af[i] for i in req_idx]
    df['assigned_amb_id']      = [aa[i] for i in req_idx]
    return df

print('solve_ilp loaded.')

## Figure 1 — Network Topology

In [ ]:
FAC_COLORS = {
    'ccp':        '#e74c3c',
    'r1_alpha':   '#2980b9',
    'r1_bravo':   '#27ae60',
    'r1_charlie': '#8e44ad',
    'r1_delta':   '#e67e22',
}
AMB_COLORS = {'amb_1': '#1abc9c', 'amb_2': '#f39c12'}


def draw_network_bg(ax, positions, facilities, distance_matrix, alpha=1.0):
    """Draw facility nodes and CCP→R1 edges."""
    ccp_pos = np.array(positions['ccp'])

    # Edges
    for fac in facilities:
        if fac.loc_id == 'ccp': continue
        fac_pos = np.array(positions[fac.loc_id])
        dist    = distance_matrix.loc['ccp', fac.loc_id]
        ax.plot([ccp_pos[0], fac_pos[0]], [ccp_pos[1], fac_pos[1]],
                color='#bdc3c7', lw=1.5, ls='--', zorder=1, alpha=alpha)
        mid = (ccp_pos + fac_pos) / 2
        ax.text(mid[0], mid[1], f'{dist:.0f} min', fontsize=7.5, color='#7f8c8d',
                ha='center', va='bottom',
                bbox=dict(fc='white', ec='none', alpha=0.7, pad=1))

    # Nodes
    for fac in facilities:
        pos   = np.array(positions[fac.loc_id])
        color = FAC_COLORS.get(fac.loc_id, '#95a5a6')
        size  = 900 if fac.loc_id == 'ccp' else 550
        ax.scatter(*pos, s=size, c=color, zorder=5,
                   edgecolors='white', linewidths=2.5, alpha=alpha)

        if fac.loc_id == 'ccp':
            label = 'CCP\n10 casualties'
        else:
            ewt   = list(fac.units.values())[0]._wait if fac.units else 0
            label = f'{fac.loc_id}\nEWT {ewt:.0f} min'
        ax.annotate(label, pos,
                    textcoords='offset points', xytext=(0, 14),
                    ha='center', fontsize=8, fontweight='bold', color=color)


fig, ax = plt.subplots(figsize=(13, 7))
draw_network_bg(ax, POSITIONS, ALL_FACILITIES, distance_matrix)

# Ambulances
offsets = {'amb_1': (-0.3, -0.6), 'amb_2': (0.3, -0.6)}
for amb in ALL_AMBULANCES:
    pos    = np.array(POSITIONS[amb.location.loc_id])
    color  = AMB_COLORS[amb.amb_id]
    off    = offsets[amb.amb_id]
    ax.scatter(pos[0]+off[0], pos[1]+off[1], s=350, c=color, marker='*',
               zorder=6, edgecolors='white', linewidths=1.5)
    ax.annotate(f'{amb.amb_id} (cap={amb.capacity})',
                (pos[0]+off[0], pos[1]+off[1]),
                textcoords='offset points', xytext=(0, -16),
                ha='center', fontsize=8, color=color, fontweight='bold')

# Legend — placed outside axes to the right
legend_handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=12, label='CCP'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#2980b9', markersize=10, label='R1 facility'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='#1abc9c', markersize=12, label='amb_1  (starts at CCP)'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='#f39c12', markersize=12, label='amb_2  (starts at r1_alpha)'),
    Line2D([0],[0], color='#bdc3c7', lw=1.5, ls='--', label='eligible route'),
]
ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(1.02, 1),
          borderaxespad=0, fontsize=9, framealpha=0.9)
ax.set_title('Network Topology  |  distances = travel time (min)  |  EWT = expected wait if joined now',
             fontsize=11, fontweight='bold', pad=12)
ax.set_aspect('equal')
ax.axis('off')
plt.tight_layout()
plt.show()

## Run All Policy Configurations

In [ ]:
CONFIGS = [
    dict(label='Triage only\n(α=1,β=0,γ=0,δ=0)',         alpha=1.0, beta=0.0,  gamma=0.0,  delta=0.0),
    dict(label='+ Sojourn\n(α=1,β=1,γ=0,δ=0)',            alpha=1.0, beta=1.0,  gamma=0.0,  delta=0.0),
    dict(label='+ Transport\n(α=1,β=0,γ=1,δ=0)',          alpha=1.0, beta=0.0,  gamma=1.0,  delta=0.0),
    dict(label='+ EWT\n(α=1,β=0,γ=0,δ=1)',                alpha=1.0, beta=0.0,  gamma=0.0,  delta=1.0),
    dict(label='Balanced\n(α=1,β=1,γ=0.5,δ=0.5)',         alpha=1.0, beta=1.0,  gamma=0.5,  delta=0.5),
    dict(label='Triage+Travel+EWT\n(α=1,β=0.01,γ=1,δ=1)', alpha=1.0, beta=0.01, gamma=1.0,  delta=1.0),
]

RESULTS = {}
for cfg in CONFIGS:
    reset_scenario()
    r = run_c2_cycle(
        env, ALL_REQUESTS,
        lever_fac_elig=0, lever_amb_elig=0,
        alpha=cfg['alpha'], beta=cfg['beta'],
        gamma=cfg['gamma'], delta=cfg['delta'],
    )
    RESULTS[cfg['label']] = r['assigned_df'].copy()
    n_assigned = r['assigned_df']['assigned_amb_id'].notna().sum()
    print(f"{cfg['label'].replace(chr(10),' '):<50}  →  {n_assigned}/4 assigned')")

## Figure 2 — Assignment Maps

In [ ]:
TRIAGE_COLORS = {'T1': '#e74c3c', 'T2': '#f39c12', 'T3': '#2ecc71'}

ncols = 3
nrows = -(-len(CONFIGS) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 6.5))
axes = axes.flatten()

for i, cfg in enumerate(CONFIGS):
    label    = cfg['label']
    ax       = axes[i]
    df       = RESULTS[label]
    assigned = df.dropna(subset=['assigned_amb_id'])

    ccp_pos = np.array(POSITIONS['ccp'])
    for fac in ALL_FACILITIES:
        if fac.loc_id == 'ccp': continue
        fp = np.array(POSITIONS[fac.loc_id])
        ax.plot([ccp_pos[0], fp[0]], [ccp_pos[1], fp[1]],
                color='#ecf0f1', lw=2, zorder=1)

    for (amb_id, dest_id), grp in assigned.groupby(['assigned_amb_id', 'assigned_facility_id']):
        dest_pos = np.array(POSITIONS[dest_id])
        n_pts    = len(grp)
        color    = AMB_COLORS.get(amb_id, '#7f8c8d')
        rad      = 0.15 if amb_id == 'amb_1' else -0.15
        ax.annotate('',
                    xy=dest_pos * 0.85,
                    xytext=ccp_pos,
                    arrowprops=dict(
                        arrowstyle=f'->, head_width=0.3, head_length=0.2',
                        color=color, lw=1.5 + n_pts * 0.8,
                        connectionstyle=f'arc3,rad={rad}',
                    ))

    for fac in ALL_FACILITIES:
        pos   = np.array(POSITIONS[fac.loc_id])
        color = FAC_COLORS.get(fac.loc_id, '#95a5a6')
        if fac.loc_id == 'ccp':
            n_here = df['assigned_amb_id'].isna().sum()
            ax.scatter(*pos, s=700, c=color, zorder=5, edgecolors='white', linewidths=2)
            ax.annotate(f'CCP\n({n_here} pending)', pos,
                        textcoords='offset points', xytext=(0, 14),
                        ha='center', fontsize=8, fontweight='bold', color=color)
        else:
            here = assigned[assigned['assigned_facility_id'] == fac.loc_id]
            n    = len(here)
            ax.scatter(*pos, s=450, c=color, zorder=5, edgecolors='white', linewidths=2,
                       alpha=(1.0 if n > 0 else 0.35))
            if n > 0:
                tc = ' '.join([f'{t}:{v}' for t, v in here['triage'].value_counts().items()])
                ax.annotate(f'{fac.loc_id}\n{tc}', pos,
                            textcoords='offset points', xytext=(0, 13),
                            ha='center', fontsize=8, fontweight='bold', color=color)
            else:
                ax.annotate(fac.loc_id, pos,
                            textcoords='offset points', xytext=(0, 11),
                            ha='center', fontsize=7, color=color, alpha=0.5)

    offsets = {'amb_1': (-0.25, -0.5), 'amb_2': (0.25, -0.5)}
    for amb in ALL_AMBULANCES:
        p   = np.array(POSITIONS[amb.location.loc_id])
        off = offsets[amb.amb_id]
        ax.scatter(p[0]+off[0], p[1]+off[1], s=200, c=AMB_COLORS[amb.amb_id],
                   marker='*', zorder=6, edgecolors='white')

    ax.set_title(label, fontsize=9, fontweight='bold', pad=8)
    ax.set_aspect('equal')
    ax.axis('off')

for j in range(len(CONFIGS), len(axes)):
    axes[j].set_visible(False)

leg_els = [
    Line2D([0],[0], color=AMB_COLORS['amb_1'], lw=3, label='amb_1 route'),
    Line2D([0],[0], color=AMB_COLORS['amb_2'], lw=3, label='amb_2 route'),
    Line2D([0],[0], marker='*', color='w', markerfacecolor='gray', markersize=10, label='ambulance start'),
]
fig.legend(handles=leg_els, loc='lower center', ncol=3, fontsize=9,
           bbox_to_anchor=(0.5, -0.01), framealpha=0.9)
fig.suptitle('Assignment Maps by Configuration', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Figure 3 — Patient × Policy Heatmap

In [ ]:
DEST_COLOR_MAP = {
    'r1_alpha':   '#2980b9',
    'r1_bravo':   '#27ae60',
    'r1_charlie': '#8e44ad',
    'r1_delta':   '#e67e22',
    '—':          '#ecf0f1',
}

config_labels = [cfg['label'] for cfg in CONFIGS]

fig, ax = plt.subplots(figsize=(15, 6))

for col_idx, label in enumerate(config_labels):
    df = RESULTS[label]
    for row_idx, req in enumerate(ALL_REQUESTS):
        row  = df[df['req_id'] == req.req_id].iloc[0]
        dest = row['assigned_facility_id'] if pd.notna(row['assigned_facility_id']) else '—'
        col  = DEST_COLOR_MAP.get(dest, '#ecf0f1')
        rect = plt.Rectangle((col_idx, row_idx), 1, 1, color=col, ec='white', lw=2)
        ax.add_patch(rect)
        short   = dest.replace('r1_', '') if dest != '—' else '—'
        txt_col = 'white' if dest != '—' else '#aaa'
        ax.text(col_idx + 0.5, row_idx + 0.5, short,
                ha='center', va='center', fontsize=8, color=txt_col, fontweight='bold')

ax.set_xlim(0, len(config_labels))
ax.set_ylim(0, len(ALL_REQUESTS))
ax.set_xticks([i + 0.5 for i in range(len(config_labels))])
ax.set_xticklabels([lbl.replace('\n', ' ') for lbl in config_labels],
                   rotation=25, ha='right', fontsize=9)
ax.set_yticks([i + 0.5 for i in range(len(ALL_REQUESTS))])
ax.set_yticklabels(
    [f"{req.req_id}  {req.triage}  t={req.t_issued:.0f}" for req in ALL_REQUESTS],
    fontsize=9,
)
ax.invert_yaxis()

triage_bands = [('T3', 0, 3, '#eafaf1'), ('T2', 3, 7, '#fef5e7'), ('T1', 7, 10, '#fde8e8')]
for _, y0, y1, color in triage_bands:
    ax.axhspan(y0, y1, xmin=-0.05, xmax=0, color=color, clip_on=False, lw=0)

patches = [mpatches.Patch(color=v, label=k) for k, v in DEST_COLOR_MAP.items()]
ax.legend(handles=patches, bbox_to_anchor=(1.01, 1), loc='upper left',
          title='Destination', fontsize=9)
ax.set_title('Assigned Destination  ·  Patients (rows) × Configurations (columns)',
             fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

## Figure 4 — Destination Distribution (Stacked Bars)

In [ ]:
DEST_ORDER   = ['r1_alpha', 'r1_bravo', 'r1_charlie', 'r1_delta', 'unassigned']
TRIAGE_ORDER = ['T1', 'T2', 'T3']
BAR_COLORS   = {dest: DEST_COLOR_MAP.get(dest, '#ecf0f1') for dest in DEST_ORDER}
BAR_COLORS['unassigned'] = '#bdc3c7'

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4.5), sharey=True)
axes = axes.flatten()

for i, cfg in enumerate(CONFIGS):
    label = cfg['label']
    ax    = axes[i]
    df    = RESULTS[label].copy()
    df['dest'] = df['assigned_facility_id'].fillna('unassigned')

    bottoms = {dest: 0 for dest in DEST_ORDER}
    for triage in TRIAGE_ORDER:
        vals = [len(df[(df['dest'] == d) & (df['triage'] == triage)]) for d in DEST_ORDER]
        bars = ax.bar(DEST_ORDER, vals,
                      bottom=[bottoms[d] for d in DEST_ORDER],
                      color=TRIAGE_COLORS[triage], edgecolor='white', lw=1.5,
                      label=triage)
        for bar, val in zip(bars, vals):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_y() + bar.get_height() / 2,
                        str(val), ha='center', va='center',
                        fontsize=9, fontweight='bold', color='white')
        for d, v in zip(DEST_ORDER, vals):
            bottoms[d] += v

    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xticks(range(len(DEST_ORDER)))
    ax.set_xticklabels([d.replace('r1_', 'R1-') for d in DEST_ORDER],
                       rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Patients')
    ax.set_ylim(0, 6)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    if i == 0:
        ax.legend(title='Triage', fontsize=8, loc='upper right')

for j in range(len(CONFIGS), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Destination Distribution by Configuration  ·  bar color = triage level',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()